# 03 – Model Training and Evaluation

This notebook trains the selected Gradient Boosting Regressor using the preprocessor from the preceding notebook. It reproduces the earlier train/test split (`random_state=42`) and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path


In [ ]:
df = pd.read_csv("../Insurance.csv").drop_duplicates().copy()
X = df.drop(columns=["charges"])
y = df["charges"]
num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ]
)
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

In [ ]:
model = GradientBoostingRegressor(random_state=42)
model.fit(X_train_final, y_train)
y_pred = model.predict(X_test_final)

def evaluate(y_true, y_pred, p):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    adjusted_r2 = 1 - ((1-r2)*(n-1)/(n-p-1))
    return pd.DataFrame([{
        "MAE": mae, "MSE": mse, "RMSE": rmse,
        "R2": r2, "Adjusted_R2": adjusted_r2
    }])

results = evaluate(y_test, y_pred, X_test_final.shape[1])
results

## Save the trained model and fitted preprocessor

Run this cell from the notebook. The files are saved in the project's `models/` folder. The Streamlit app loads these exact files.

In [ ]:
Path("../models").mkdir(exist_ok=True)
joblib.dump(model, "../models/insurance_model.pkl")
joblib.dump(preprocessor, "../models/preprocessor.pkl")
print("Saved model and preprocessor.")